### **Day 8**

In [1]:
import torch
import torch.nn.functional as F

def lift_consistency_loss(cl_pred, cp_pred):
    """
    Approximate lift from Cp.
    Simple surrogate integration.
    """

    cp_lift = torch.mean(cp_pred, dim=1)

    return F.mse_loss(cl_pred.squeeze(), cp_lift)

In [2]:
def cp_smoothness_loss(cp_pred):
    diff = cp_pred[:,1:] - cp_pred[:,:-1]

    return torch.mean(diff**2)

In [3]:
def drag_positive_loss(cd_pred):
    return torch.mean(
        torch.relu(-cd_pred)
    )

In [4]:
def pinn_loss(
    cl_pred,
    cd_pred,
    cp_pred,
    cl_true,
    cd_true,
    cp_true,
):

    data_loss = (
        F.mse_loss(cl_pred, cl_true)
        +
        F.mse_loss(cd_pred, cd_true)
        +
        F.mse_loss(cp_pred, cp_true)
    )

    lift_loss = lift_consistency_loss(
        cl_pred,
        cp_pred
    )

    smooth_loss = cp_smoothness_loss(
        cp_pred
    )

    drag_loss = drag_positive_loss(
        cd_pred
    )

    total_loss = (
        data_loss
        + 1.0*lift_loss
        + 0.01*smooth_loss
        + 0.1*drag_loss
    )

    return total_loss

In [5]:
batch = 32

cl_pred = torch.randn(batch,1)
cd_pred = torch.randn(batch,1)
cp_pred = torch.randn(batch,98)

cl_true = torch.randn(batch,1)
cd_true = torch.randn(batch,1)
cp_true = torch.randn(batch,98)

loss = pinn_loss(
    cl_pred,
    cd_pred,
    cp_pred,
    cl_true,
    cd_true,
    cp_true
)

print(loss)

tensor(8.2602)
